El agente supervisor es un proceso de monitorización de corrida que agrega el estado de Agent01, Agent02 y Agent03 a
  nivel de RUN_ID. Su función es evaluar salud operativa, progreso, bloqueos de cierre y anomalías estructurales sin
  inspeccionar stdout de los procesos, sino consumiendo artefactos persistidos en RUN_DIR.

  Fuentes monitorizadas:

  - download_live_status.json
      - progreso de descarga (tasks_total, done_ok, done_bad, pending, updated_utc)
  - live_status_quotes_strict.json
      - estado de validación estricta (files_current_snapshot, files_pending, retry_pending_files_current,
        severity_counts_current, top_causes_current, updated_utc)
  - agent03_outputs/run_summary.json
      - estado consolidado de cobertura/cierre (gate_status, retry_pending_files, hard_fail_files, mean_coverage_ok)

  Controles implementados:

  - stall detection
      - detecta artefactos sin actualización por encima de StallSec
  - download no-progress detection
      - detecta ausencia de incremento en done_ok + done_bad con pending > 0
  - hard fail spike detection
      - detecta incrementos bruscos de HARD_FAIL entre snapshots
  - retry spike detection
      - detecta incrementos bruscos de retry_pending_files_current
  - structural validation alert
      - eleva alerta si top_causes_current contiene parquet_unreadable
  - gate monitoring
      - clasifica el estado de cierre reportado por Agent03

  Clasificación de salida:

  - RUN STATE
      - estados esperables durante ejecución
      - ejemplos:
          - AGENT01_RUNNING
          - AGENT02_RETRY_OPEN
          - AGENT03_GATE_OPEN: NO_CLOSE_RETRY_PENDING
  - WARNINGS
      - condiciones relevantes no necesariamente críticas
      - ejemplos:
          - AGENT02_HARD_PRESENT
          - AGENT02_HARD_FAIL_SPIKE
          - AGENT02_RETRY_SPIKE
          - AGENT03_GATE_BLOCKED: NO_CLOSE_HARD_FAIL
  - ALERTS
      - anomalías operativas o estructurales
      - ejemplos:
          - AGENT01_STALL
          - AGENT01_NO_PROGRESS
          - AGENT02_STALL
          - AGENT02_STRUCTURAL: parquet_unreadable
          - AGENT03_STALL

  Parámetros operativos:

  - RunId
  - IntervalSec
  - StallSec
  - HardFailSpike
  - RetrySpike
  - DownloadNoProgressSec
  - HardFailWarnThreshold
  - BeepOnAlert

  Resultado:

  - consolidación de observabilidad por corrida
  - separación entre estado esperado, degradación operativa y fallo crítico
  - soporte para decisión de continuidad, retry o cierre de run


---

Ahora el supervisor deja histórico persistente por RUN_ID en el mismo RUN_DIR.

Artefactos nuevos que crea:

- supervisor_events_history.csv
- supervisor_current_status.json

Qué guarda:

- supervisor_events_history.csv
    - histórico acumulado de cambios de estado
    - columnas:
        - observed_at_local
        - observed_at_utc
        - run_id
        - run_dir
        - level
        - message
    - niveles:
        - RUN_STATE
        - WARNING
        - ALERT
- supervisor_current_status.json
    - snapshot actual consolidado
    - incluye:
        - run_state
        - warnings
        - alerts
        - resumen de Agent01
        - resumen de Agent02
        - resumen de Agent03

Criterio de escritura:

- el CSV histórico se actualiza cuando cambia el conjunto de RUN STATE, WARNINGS o ALERTS
- el JSON actual se reescribe en cada ciclo

Ya está validado sobre el run actual y ambos archivos aparecen correctamente.

Lanzadera:
```
powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent_supervisor.ps1" -RunId "20260312_quotes_preprod30_full_lifecycle" -IntervalSec 10 -StallSec 180 -HardFailWarnThreshold 100 -BeepOnAlert
```

Si quieres, el siguiente paso es añadir una tercera salida:

- supervisor_alerts_only.csv
solo con WARNING y ALERT, para revisión rápida de incidencias.

In [ ]:
agent-supervisor
time : 2026-03-12 18:39:28
run  : 20260312_quotes_preprod30_full_lifecycle
dir  : C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_preprod30_full_lifecycle
----------------------------------------------------------------------------------------------------
AGENT01 download
  updated_utc      : 2026-03-12T17:39:18.446305+00:00
  tasks_total      : 65676
  done_ok          : 7796
  done_bad         : 0
  pending          : 57880
  concurrent       : 24
  session          : 04:00:00-20:00:00
  age_sec          : 10

AGENT02 strict validation
  updated_utc      : 2026-03-12T17:39:27.801169+00:00
  files_snapshot   : 7812
  files_pending    : 618
  processed_state  : 329
  retry_pending    : 2863
  hard_fail        : 449
  age_sec          : 1
  top_causes_current:
    - crossed_rows_present_but_under_threshold: 4157
    - crossed_ratio_gt_threshold: 449
    - crossed_ratio_gt_hard_cap: 59

AGENT03 coverage summary
  gate_status      : NO_CLOSE_RETRY_PENDING
  mean_coverage_ok : 0.0
  retry_pending    : 42
  hard_fail_files  : 0
  file_age_sec     : 15

files
  download_events_current exists : True
  events_current exists          : True
  retry_current exists           : True
  supervisor_events exists       : True
  supervisor_current exists      : True

RUN STATE:
  - AGENT01_RUNNING: descarga aun en progreso (pending=57880)
  - AGENT02_RETRY_OPEN: retry_pending_files_current=2863
  - AGENT03_GATE_OPEN: NO_CLOSE_RETRY_PENDING

WARNINGS:
  - AGENT02_HARD_PRESENT: hard_fail_current=449

ALERTS: none